In [2]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time, os

uploaded = files.upload()  # select all six CSVs
spark = SparkSession.builder.appName("StreamingAssessment").getOrCreate()
def read_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv


In [3]:
sessions_df = read_csv("watch_sessions_80000.csv")
sessions_extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------

**Part A - DataFrame operations**

01. Display the schema, row count, and first 10 records of sessions_df. Identify numeric, string, and timestamp
fields.

In [4]:
sessions_df.printSchema()
print("Row count:", sessions_df.count())
sessions_df.show(10, truncate=False)

numeric_types = ("int", "bigint", "double", "float", "decimal", "long", "short")
print("Numeric  :", [c for c, t in sessions_df.dtypes if t.startswith(numeric_types)])
print("String   :", [c for c, t in sessions_df.dtypes if t == "string"])
print("Timestamp:", [c for c, t in sessions_df.dtypes if t in ("timestamp", "date")])

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Row count: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+---

02. Select session_id, subscriber_id, content_id, watch_minutes, and session_status. Filter completed sessions with
watch_minutes greater than 60.

In [5]:
selected_df = sessions_df.select("session_id", "subscriber_id", "content_id", "watch_minutes", "session_status")
long_completed = selected_df.filter((F.col("session_status") == "Completed") & (F.col("watch_minutes") > 60))
print(long_completed.count()); long_completed.show(10)

3403
+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|     SUB01859|    CT0075|          125|     Completed|
|WS00000179|     SUB00935|    CT0054|           62|     Completed|
|WS00000184|     SUB00072|    CT0119|           91|     Completed|
|WS00000222|     SUB01603|    CT0103|          116|     Completed|
|WS00000233|     SUB01233|    CT0027|          158|     Completed|
|WS00000244|     SUB00985|    CT0048|          140|     Completed|
|WS00000249|     SUB00681|    CT0100|          120|     Completed|
|WS00000257|     SUB02115|    CT0120|           68|     Completed|
|WS00000292|     SUB01068|    CT0048|          141|     Completed|
|WS00000326|     SUB00368|    CT0096|           97|     Completed|
+----------+-------------+----------+-------------+--------------+
only showing top 10 rows


03. Create watch_hours from watch_minutes and a high_engagement flag for sessions with at least 75% completion.
Preserve null completion values.

In [6]:
eng_df = (sessions_df
    .withColumn("watch_hours", F.col("watch_minutes") / 60)
    .withColumn("high_engagement",
        F.when(F.col("completion_pct").isNull(), F.lit(None).cast("boolean"))
         .when(F.col("completion_pct") >= 75, True)
         .otherwise(False)))
eng_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)
eng_df.groupBy("high_engagement").count().show()

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|        watch_hours|completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|            6|                0.1|          11.8|          false|
|WS00000002|           31| 0.5166666666666667|          56.4|          false|
|WS00000003|           29|0.48333333333333334|          63.0|          false|
|WS00000004|           69|               1.15|          75.0|           true|
|WS00000005|           36|                0.6|          23.4|          false|
|WS00000006|           20| 0.3333333333333333|          19.8|          false|
|WS00000007|           78|                1.3|          83.9|           true|
|WS00000008|           22|0.36666666666666664|          66.7|          false|
|WS00000009|           21|               0.35|          80.8|           true|
|WS00000010|           21|               0.35|          65.6|   

04. Sort sessions by data_used_mb descending and display the top 20 records.

In [7]:
sessions_df.orderBy(F.col("data_used_mb").desc()).show(20)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

**Part B - Data quality and conditional transformations**

05. Count missing values in subscriber_id, device_id, completion_pct, buffering_seconds, and user_rating.

In [8]:
null_cols = ["subscriber_id", "device_id", "completion_pct", "buffering_seconds", "user_rating"]
sessions_df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls") for c in null_cols]).show()

+-------------------+---------------+--------------------+-----------------------+-----------------+
|subscriber_id_nulls|device_id_nulls|completion_pct_nulls|buffering_seconds_nulls|user_rating_nulls|
+-------------------+---------------+--------------------+-----------------------+-----------------+
|                951|           2003|                3513|                   4424|            77605|
+-------------------+---------------+--------------------+-----------------------+-----------------+



06. Create a new DataFrame that replaces missing buffering_seconds with zero. Do not modify the original
DataFrame.

In [9]:
filled_df = sessions_df.fillna({"buffering_seconds": 0})
print("Filled nulls:", filled_df.filter(F.col("buffering_seconds").isNull()).count())
print("Original nulls (unchanged):", sessions_df.filter(F.col("buffering_seconds").isNull()).count())

Filled nulls: 0
Original nulls (unchanged): 4424


07. Create another DataFrame that excludes sessions with missing device_id. Compare record counts.

In [12]:
with_missing_device_df = sessions_df.filter(F.col("device_id").isNotNull())
a, b = sessions_df.count(), with_missing_device_df.count()
print(f"Original: {a} | With missing_device_id: {b} | Excluded: {a - b}")

Original: 80000 | With missing_device_id: 77997 | Excluded: 2003


08. Use when() and otherwise() to label completion_pct as Low (below 30), Medium (30 to below 90), High (90 or
more), or Unknown (null).

In [13]:
band_df = sessions_df.withColumn("completion_band",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when(F.col("completion_pct") < 90, "Medium")
     .otherwise("High"))
band_df.groupBy("completion_band").count().show()

+---------------+-----+
|completion_band|count|
+---------------+-----+
|           High| 6117|
|            Low|13267|
|        Unknown| 3513|
|         Medium|57103|
+---------------+-----+



**Part C - Joins and aggregations**

In [15]:
subs_dim    = subscribers_df.select("subscriber_id", "region", "plan_id", "signup_date", "subscriber_status")
content_dim = content_df.select("content_id", "content_title", "genre")
device_dim  = devices_df.select("device_id", "device_type")
plans_dim   = plans_df.select("plan_name", "monthly_fee_inr")

09. Join sessions with the content catalog and calculate session counts and total watch_minutes by genre.

In [16]:
sc_df = sessions_df.join(content_dim, on="content_id", how="inner")
(sc_df.groupBy("genre")
    .agg(F.count("*").alias("session_count"), F.sum("watch_minutes").alias("total_watch_minutes"))
    .orderBy(F.desc("total_watch_minutes")).show())

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|  Adventure|        15561|             839700|
|      Drama|        10733|             519872|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
|     Sports|         9967|             383044|
|Documentary|         7432|             256626|
|   Thriller|         5972|             248430|
+-----------+-------------+-------------------+



10. Join sessions with the device master and calculate average buffering_seconds and average watch_minutes by
device_type.

In [17]:
sd_df = sessions_df.join(device_dim, on="device_id", how="inner")
(sd_df.groupBy("device_type")
    .agg(F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
         F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes"))
    .orderBy("device_type").show())

+---------------+---------------------+-----------------+
|    device_type|avg_buffering_seconds|avg_watch_minutes|
+---------------+---------------------+-----------------+
|  Android Phone|                35.68|            45.09|
|Desktop Browser|                36.03|            45.57|
|   Game Console|                35.51|            45.04|
| Laptop Browser|                36.93|            45.84|
|       Smart TV|                36.67|            45.34|
|Streaming Stick|                36.32|            45.09|
|         Tablet|                35.24|            45.39|
|         iPhone|                35.76|            45.23|
+---------------+---------------------+-----------------+



11. Join sessions, subscribers, and plans. Calculate completion rate (completed sessions / all sessions) by
plan_name, excluding null subscriber IDs.

In [19]:
ssp_df = (sessions_df.filter(F.col("subscriber_id").isNotNull())
    .join(subs_dim, on="subscriber_id", how="inner")
    .join(plans_df, on="plan_id", how="inner"))

(ssp_df.groupBy("plan_name")
    .agg(F.count("*").alias("total_sessions"),
         F.sum((F.col("session_status") == "Completed").cast("int")).alias("completed_sessions"),
         F.round(F.avg((F.col("session_status") == "Completed").cast("int")), 4).alias("completion_rate"))
    .orderBy("plan_name").show())

+---------+--------------+------------------+---------------+
|plan_name|total_sessions|completed_sessions|completion_rate|
+---------+--------------+------------------+---------------+
|    Basic|         25086|              2006|           0.08|
|   Mobile|         17510|              1429|         0.0816|
|  Premium|         12198|               975|         0.0799|
| Standard|         23684|              1868|         0.0789|
+---------+--------------+------------------+---------------+



12. Using subscriber_master and subscription_plans only, estimate monthly recurring revenue by plan for Active
subscribers. Count each subscriber once; do not multiply fees by viewing sessions.

In [22]:
active_subs = (subs_dim.filter(F.col("subscriber_status") == "Active")
               .dropDuplicates(["subscriber_id"]))

mrr_df = (active_subs.join(plans_df, on="plan_id", how="inner")
    .groupBy("plan_name")
    .agg(F.countDistinct("subscriber_id").alias("active_subscribers"),
         F.sum("monthly_fee_inr").alias("monthly_recurring_revenue")))

mrr_df.orderBy("plan_name").show()
mrr_df.agg(F.sum("monthly_recurring_revenue").alias("total_MRR")).show()

+---------+------------------+-------------------------+
|plan_name|active_subscribers|monthly_recurring_revenue|
+---------+------------------+-------------------------+
|    Basic|               632|                   157368|
|   Mobile|               443|                    66007|
|  Premium|               326|                   211574|
| Standard|               604|                   240996|
+---------+------------------+-------------------------+

+---------+
|total_MRR|
+---------+
|   675945|
+---------+



13. Identify sessions whose non-null subscriber_id or device_id has no matching master record. Report unmatched
counts for each key.

In [23]:
unmatched_subs = (sessions_df.filter(F.col("subscriber_id").isNotNull())
                  .join(subs_dim.select("subscriber_id"), on="subscriber_id", how="left_anti"))
unmatched_dev  = (sessions_df.filter(F.col("device_id").isNotNull())
                  .join(device_dim.select("device_id"), on="device_id", how="left_anti"))
print("Sessions with unmatched subscriber_id:", unmatched_subs.count())
print("Sessions with unmatched device_id    :", unmatched_dev.count())

Sessions with unmatched subscriber_id: 571
Sessions with unmatched device_id    : 481


**Part D - Date and window functions**

14. Convert session_start to timestamp. Calculate monthly session counts and total watch_minutes for 2026.

In [24]:
ts_df = (sessions_df
    .withColumn("session_ts", F.to_timestamp("session_start"))
    .withColumn("session_date", F.to_date("session_ts")))
(ts_df.filter(F.year("session_ts") == 2026)
    .groupBy(F.month("session_ts").alias("month"))
    .agg(F.count("*").alias("session_count"), F.sum("watch_minutes").alias("total_watch_minutes"))
    .orderBy("month").show())

+-----+-------------+-------------------+
|month|session_count|total_watch_minutes|
+-----+-------------+-------------------+
|    1|         9070|             407215|
|    2|         8218|             372726|
|    3|         9019|             403971|
|    4|         8817|             399432|
|    5|         9006|             407386|
|    6|         8776|             404203|
|    7|         9204|             416104|
|    8|         8996|             407374|
|    9|         8894|             406105|
+-----+-------------+-------------------+



15. Join subscriber signup dates and calculate days between signup_date and each viewing session. Display the
earliest 10 viewing sessions.

In [25]:
signup_df = (ts_df.filter(F.col("subscriber_id").isNotNull())
    .join(subs_dim.select("subscriber_id", "signup_date"), on="subscriber_id", how="inner")
    .withColumn("days_since_signup", F.datediff("session_date", F.to_date("signup_date"))))
(signup_df.select("session_id", "subscriber_id", "signup_date", "session_ts", "days_since_signup")
    .orderBy("session_ts").show(10, truncate=False))

+----------+-------------+-----------+-------------------+-----------------+
|session_id|subscriber_id|signup_date|session_ts         |days_since_signup|
+----------+-------------+-----------+-------------------+-----------------+
|WS00035235|SUB01707     |2024-02-13 |2026-01-01 00:00:00|688              |
|WS00074739|SUB02206     |2025-08-31 |2026-01-01 00:04:00|123              |
|WS00032058|SUB01248     |2024-03-17 |2026-01-01 00:04:00|655              |
|WS00048560|SUB02342     |2025-08-25 |2026-01-01 00:14:00|129              |
|WS00067087|SUB00563     |2024-08-11 |2026-01-01 00:15:00|508              |
|WS00004245|SUB00711     |2024-10-26 |2026-01-01 00:18:00|432              |
|WS00041710|SUB00261     |2025-05-15 |2026-01-01 00:21:00|231              |
|WS00045132|SUB02167     |2025-11-22 |2026-01-01 00:28:00|40               |
|WS00036438|SUB01237     |2024-02-01 |2026-01-01 00:41:00|700              |
|WS00030050|SUB00646     |2024-11-20 |2026-01-01 00:51:00|407              |

16. Classify sessions as Weekday or Weekend using the session date, then compare average watch_minutes.

In [26]:
# dayofweek: 1 = Sunday ... 7 = Saturday
dow_df = ts_df.withColumn("day_type",
    F.when(F.dayofweek("session_date").isin(1, 7), "Weekend").otherwise("Weekday"))
(dow_df.groupBy("day_type")
    .agg(F.count("*").alias("sessions"), F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes")).show())

+--------+--------+-----------------+
|day_type|sessions|avg_watch_minutes|
+--------+--------+-----------------+
| Weekday|   57228|            45.26|
| Weekend|   22772|            45.41|
+--------+--------+-----------------+



17. Aggregate total watch_minutes by genre and content_title; use dense_rank() to identify the top three titles
within each genre.

In [27]:
total_watch_minutes_df = (sc_df.groupBy("genre", "content_title")
             .agg(F.sum("watch_minutes").alias("total_watch_minutes")))
w17 = Window.partitionBy("genre").orderBy(F.desc("total_watch_minutes"))
(total_watch_minutes_df.withColumn("rank", F.dense_rank().over(w17))
    .filter(F.col("rank") <= 3).orderBy("genre", "rank").show())

+-----------+----------------+-------------------+----+
|      genre|   content_title|total_watch_minutes|rank|
+-----------+----------------+-------------------+----+
|  Adventure|Infinite Horizon|              66395|   1|
|  Adventure|    Lost Journey|              61823|   2|
|  Adventure|    Solar Memory|              60540|   3|
|  Animation|      Last River|              51398|   1|
|  Animation| Emerald Circuit|              51310|   2|
|  Animation|   Golden Puzzle|              50515|   3|
|     Comedy|    Blue Horizon|              61428|   1|
|     Comedy|   Bright Valley|              60921|   2|
|     Comedy|   Silver Valley|              60422|   3|
|Documentary|   Fading Signal|              47085|   1|
|Documentary|  Emerald Valley|              40819|   2|
|Documentary|   Solar Horizon|              36152|   3|
|      Drama|  Crimson Summit|              65724|   1|
|      Drama|  Rising Archive|              57301|   2|
|      Drama|     Wild Harbor|              4547

18. Use row_number() to retrieve the most recent session for each non-null subscriber_id.

In [28]:
w18 = Window.partitionBy("subscriber_id").orderBy(F.col("session_ts").desc(), F.col("session_id").desc())
latest_df = (ts_df.filter(F.col("subscriber_id").isNotNull())
    .withColumn("rn", F.row_number().over(w18)).filter(F.col("rn") == 1).drop("rn"))
print("Subscribers with sessions:", latest_df.count())
latest_df.select("subscriber_id", "session_id", "session_ts", "watch_minutes").show(10)

Subscribers with sessions: 2401
+-------------+----------+-------------------+-------------+
|subscriber_id|session_id|         session_ts|watch_minutes|
+-------------+----------+-------------------+-------------+
|     SUB00001|WS00020822|2026-09-11 19:19:00|           34|
|     SUB00002|WS00032744|2026-09-02 05:41:00|           45|
|     SUB00003|WS00006110|2026-09-30 03:13:00|           11|
|     SUB00004|WS00037660|2026-09-30 17:48:00|           62|
|     SUB00005|WS00070939|2026-09-18 02:50:00|           22|
|     SUB00006|WS00057918|2026-09-23 06:26:00|          104|
|     SUB00007|WS00024658|2026-09-23 20:45:00|           55|
|     SUB00008|WS00032330|2026-09-26 22:41:00|           41|
|     SUB00009|WS00008916|2026-09-04 22:21:00|           30|
|     SUB00010|WS00049685|2026-09-29 06:36:00|            6|
+-------------+----------+-------------------+-------------+
only showing top 10 rows


19. Use lag() to compare each subscriber's watch_minutes with their previous session, ordered by session_start and
session_id.

In [29]:
w19 = Window.partitionBy("subscriber_id").orderBy("session_ts", "session_id")
lag_df = (ts_df.filter(F.col("subscriber_id").isNotNull())
    .withColumn("prev_watch_minutes", F.lag("watch_minutes", 1).over(w19))
    .withColumn("diff_from_prev", F.col("watch_minutes") - F.col("prev_watch_minutes")))
lag_df.select("subscriber_id", "session_id", "session_ts", "watch_minutes", "prev_watch_minutes", "diff_from_prev") \
      .orderBy("subscriber_id", "session_ts").show(15)

+-------------+----------+-------------------+-------------+------------------+--------------+
|subscriber_id|session_id|         session_ts|watch_minutes|prev_watch_minutes|diff_from_prev|
+-------------+----------+-------------------+-------------+------------------+--------------+
|     SUB00001|WS00064390|2026-01-02 17:32:00|           58|              NULL|          NULL|
|     SUB00001|WS00045183|2026-01-05 04:51:00|           78|                58|            20|
|     SUB00001|WS00028484|2026-01-09 08:46:00|            4|                78|           -74|
|     SUB00001|WS00043555|2026-01-24 19:41:00|          108|                 4|           104|
|     SUB00001|WS00041862|2026-02-18 07:43:00|           79|               108|           -29|
|     SUB00001|WS00059616|2026-02-25 07:38:00|           18|                79|           -61|
|     SUB00001|WS00042345|2026-03-13 07:34:00|           44|                18|            26|
|     SUB00001|WS00009166|2026-03-17 08:59:00|    

**Part E - Union, lazy execution, caching and partitioning**

In [30]:
def timed(label, fn):
    t = time.time(); r = fn(); e = time.time() - t
    print(f"{label}: {e:.3f}s"); return r, e

20. Combine both session batches using unionByName(). Count rows and remove duplicate session_id values.
Compare the totals.

In [31]:
combined_df = sessions_df.unionByName(sessions_extra_df)
unique_df   = combined_df.dropDuplicates(["session_id"])
print("Main:", sessions_df.count(), "| Extra:", sessions_extra_df.count())
print("After union:", combined_df.count())
print("Unique session_id:", unique_df.count())

Main: 80000 | Extra: 10000
After union: 90000
Unique session_id: 89400


21. Build a lazy pipeline with filter(), join(), withColumn(), groupBy(), and orderBy(). Inspect explain() before calling
an action.

In [32]:
lazy_pipeline = (sessions_df
    .filter(F.col("session_status") == "Completed")
    .join(content_dim, on="content_id", how="inner")
    .withColumn("watch_hours", F.col("watch_minutes") / 60)
    .groupBy("genre").agg(F.sum("watch_hours").alias("total_watch_hours"))
    .orderBy(F.desc("total_watch_hours")))
lazy_pipeline.explain(True)   # nothing has executed yet
lazy_pipeline.show()          # the action triggers execution

== Parsed Logical Plan ==
'Sort ['total_watch_hours DESC NULLS LAST], true
+- Aggregate [genre#100], [genre#100, sum(watch_hours#1279) AS total_watch_hours#1280]
   +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, (cast(watch_minutes#22 as double) / cast(60 as double)) AS watch_hours#1279]
      +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100]
         +- Join Inner, (content_id#19 = content_id#98)
            :- Filter (session_status#26 = Completed)
            :  +- Relation [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,bu

22. Cache completed sessions, run count() and an aggregation twice, and compare timings. Explain when the cache
is populated.

In [33]:
completed_df = sessions_df.filter(F.col("session_status") == "Completed").cache()
agg = lambda: completed_df.groupBy("stream_quality").agg(F.sum("watch_minutes")).collect()

_, c1 = timed("count #1 (populates cache)", lambda: completed_df.count())
_, c2 = timed("count #2 (reads cache)",     lambda: completed_df.count())
_, a1 = timed("aggregation #1",             agg)
_, a2 = timed("aggregation #2",             agg)
print("is_cached:", completed_df.is_cached)
completed_df.unpersist()

count #1 (populates cache): 1.108s
count #2 (reads cache): 0.264s
aggregation #1: 0.393s
aggregation #2: 0.157s
is_cached: True


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

**When is the cache populated?** `cache()` is lazy – it only marks the plan. The first action (`count #1`) computes the data and stores the partitions; later actions read from memory. The first action is therefore usually *not* faster (it pays the caching cost), and on small data like this the gain may be small.

23. Persist sessions with stream_quality = "4K" using MEMORY_AND_DISK. Execute two actions and then call
unpersist().

In [34]:
q4k = sessions_df.filter(F.col("stream_quality") == "4K").persist(StorageLevel.MEMORY_AND_DISK)
print("4K sessions:", q4k.count()) # action 1
q4k.agg(F.sum("data_used_mb").alias("total_data_mb"),
        F.avg("buffering_seconds").alias("avg_buffering")).show() # action 2
print("Storage level:", q4k.storageLevel)
q4k.unpersist()
print("After unpersist, is_cached:", q4k.is_cached)

4K sessions: 7312
+--------------------+------------------+
|       total_data_mb|     avg_buffering|
+--------------------+------------------+
|1.6094375500000007E7|36.879112914915204|
+--------------------+------------------+

Storage level: Disk Memory Serialized 1x Replicated
After unpersist, is_cached: False


24. Repartition sessions into 8 partitions and display row counts per partition using spark_partition_id().

In [35]:
df8 = sessions_df.repartition(8)
print("Partitions:", df8.rdd.getNumPartitions())
df8.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|10000|
|           1| 9999|
|           2|10001|
|           3|10000|
|           4|10000|
|           5|10000|
|           6|10000|
|           7|10000|
+------------+-----+



25. Starting from 8 partitions, compare coalesce(2) and repartition(2) with explain(). Write a chosen output
DataFrame to Parquet and inspect the part files.

In [36]:
coalesce_2 = df8.coalesce(2)
repartition_2 = df8.repartition(2)
print("coalesce(2):", coalesce_2.rdd.getNumPartitions(), "| repartition(2):", repartition_2.rdd.getNumPartitions())
print("=== coalesce(2) plan ==="); coalesce_2.explain()
print("=== repartition(2) plan ==="); repartition_2.explain()

# Chosen output: enriched sessions
enriched_df = (sessions_df
    .join(content_dim, on="content_id", how="left")
    .join(device_dim,  on="device_id",  how="left"))
out25 = "/content/output/enriched_sessions"
enriched_df.repartition(2).write.mode("overwrite").parquet(out25)
print(sorted(os.listdir(out25)))
print("Part files:", len([f for f in os.listdir(out25) if f.startswith("part-")]))
print("Readback count:", spark.read.parquet(out25).count())

coalesce(2): 2 | repartition(2): 2
=== coalesce(2) plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 2
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3810]
            +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...
+- == Initial Plan ==
   Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3810]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#

**Final capstone - Streaming performance report**

1. Union the two batches and deduplicate by session_id.
2. Clean relevant nulls without discarding every incomplete record.
3. Join subscribers, content, devices, and plans as needed.
4. Produce monthly metrics by region and genre: session count, watch hours, completion rate, average buffering,
and total data usage.
5. Rank the top three content titles by watch hours within each genre.
6. Cache the final aggregated DataFrame, reuse it for two actions, and unpersist it.
7. Save the final report as Parquet with 3 output partitions.

In [40]:
# 1. Union + dedupe
cap = sessions_df.unionByName(sessions_extra_df).dropDuplicates(["session_id"])

# 2. Targeted null cleaning
cap = cap.filter(F.col("subscriber_id").isNotNull() & F.col("content_id").isNotNull())

# 3. Joins
cap = (cap.join(subs_dim, on="subscriber_id", how="inner")
          .join(content_dim, on="content_id", how="inner")
          .join(device_dim,  on="device_id",  how="left")
          .join(plans_df, on="plan_id", how="left"))

cap = (cap.withColumn("month", F.date_format(F.to_timestamp("session_start"), "yyyy-MM"))
          .withColumn("is_completed", (F.col("session_status") == "Completed").cast("int")))

# 4. Monthly metrics by region and genre
report = (cap.groupBy("region", "genre", "month")
    .agg(F.count("*").alias("session_count"),
         F.round(F.sum("watch_minutes") / 60, 2).alias("watch_hours"),
         F.round(F.avg("is_completed"), 4).alias("completion_rate"),
         F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
         F.round(F.sum("data_used_mb"), 2).alias("total_data_mb"))
    .orderBy("region", "genre", "month"))

# 5. Top 3 titles by watch hours within each genre
title_hours = (cap.groupBy("genre", "content_title")
    .agg(F.round(F.sum("watch_minutes") / 60, 2).alias("watch_hours")))
w_cap = Window.partitionBy("genre").orderBy(F.desc("watch_hours"))
top_titles = (title_hours.withColumn("rank", F.dense_rank().over(w_cap))
              .filter(F.col("rank") <= 3).orderBy("genre", "rank"))

# 6. Cache, reuse for two actions, unpersist
report.cache()
print("Report rows:", report.count()) # action 1
report.groupBy("region").agg(F.sum("session_count").alias("sessions"),
                             F.sum("watch_hours").alias("hours")).show() # action 2
report.show(20)
top_titles.show()

# 7. Parquet with 3 output partitions
out_rep = "/content/output/streaming_report"
report.repartition(3).write.mode("overwrite").parquet(out_rep)
top_titles.coalesce(1).write.mode("overwrite").parquet("/content/output/top_titles_by_genre")
print(sorted(os.listdir(out_rep)))
print("Part files:", len([f for f in os.listdir(out_rep) if f.startswith("part-")]))

report.unpersist()

Report rows: 360
+-------+--------+------------------+
| region|sessions|             hours|
+-------+--------+------------------+
|Central|    9630| 7278.179999999998|
|   East|   12659| 9624.540000000005|
|  North|   18574|13848.419999999996|
|  South|   24890|18816.670000000006|
|   West|   21927|16684.669999999995|
+-------+--------+------------------+

+-------+---------+-------+-------------+-----------+---------------+---------------------+-------------+
| region|    genre|  month|session_count|watch_hours|completion_rate|avg_buffering_seconds|total_data_mb|
+-------+---------+-------+-------------+-----------+---------------+---------------------+-------------+
|Central|Adventure|2026-01|          203|      182.1|         0.0739|                35.41|     188882.1|
|Central|Adventure|2026-02|          176|     168.92|         0.1136|                35.36|     184321.0|
|Central|Adventure|2026-03|          193|     168.45|         0.0518|                39.14|     186452.3|
|Cen

DataFrame[region: string, genre: string, month: string, session_count: bigint, watch_hours: double, completion_rate: double, avg_buffering_seconds: double, total_data_mb: double]